<a href="https://colab.research.google.com/github/PraiseOrly/Medibot/blob/main/Medical_Question_Answering_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **PROJECT NAME:**

# MediBot: ChatDoctor QA

---
(MediBot is a Python-based medical chatbot that answers queries using the chatdoctor_icliniq dataset from Hugging Face. It employs fuzzy string matching to find relevant question-answer pairs from ChatDoctor and iCliniq sources, delivering accurate responses via a command-line interface.)

Install necessary libraries

In [5]:
!pip install datasets
!pip install fuzzywuzzy
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which i

Import necessary libraries

In [23]:
from datasets import load_dataset
from fuzzywuzzy import fuzz
import sys
import random

Load Datasets

In [7]:
# Load the dataset
ds = load_dataset("Malikeh1375/medical-question-answering-datasets", "chatdoctor_icliniq")
data = ds['train']  # Assuming the dataset has a 'train' split

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.96k [00:00<?, ?B/s]

(…)-00000-of-00001-bd1bc39748008404.parquet:   0%|          | 0.00/4.15M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7321 [00:00<?, ? examples/s]

In [8]:
print(data[0])

{'instruction': 'Answer this question truthfully', 'input': 'Hello doctor,I had mumps five months ago and after that, I started to have an infection in my left testes. It was swollen and now it has shrunk to almost half the size of the other one. As I am sexually active, I feel a pain in each of the vas deferens after sex. If I do not have sex for days, they become sensitive. I was treated with Ceftum 500 mg, the first time I had an infection. Now my question is, is there any chance that the infection is still in my body? And, do I need to get examined for it? For the time being, please suggest some precautionary antibiotics for my relief.', 'output': 'Hello, Welcome to Chat Doctor forum. I can understand your concern. You had mumps and this is a viral infection known to cause an inflammation of the testis in some cases. Take care. For more information consult a sexologist online'}


In [14]:
# Load the dataset
ds = load_dataset("Malikeh1375/medical-question-answering-datasets", "chatdoctor_icliniq")
data = ds['train']  # Using the 'train' split

# Debug: Inspect dataset structure
print("Dataset type:", type(data))
print("First item:", data[0])
print("Column names:", data.column_names)

# Print rows 0 to 9 of the dataset
print("\nRows 0 to 9 of the chatdoctor_icliniq dataset:")
try:
    for i, item in enumerate(data.select(range(10))):
        if not isinstance(item, dict):
            print(f"Row {i}: Error - Item is not a dictionary, got {type(item)}: {item}")
            continue
        input_truncated = item['input'][:100] + ("..." if len(item['input']) > 100 else "")
        output_truncated = item['output'][:100] + ("..." if len(item['output']) > 100 else "")
        print(f"\nRow {i}:")
        print(f"Instruction: {item['instruction']}")
        print(f"Input: {input_truncated}")
        print(f"Output: {output_truncated}")
except KeyError as e:
    print(f"Error: Key not found in dataset item - {e}")
except TypeError as e:
    print(f"Error: Type mismatch - {e}")

Dataset type: <class 'datasets.arrow_dataset.Dataset'>
First item: {'instruction': 'Answer this question truthfully', 'input': 'Hello doctor,I had mumps five months ago and after that, I started to have an infection in my left testes. It was swollen and now it has shrunk to almost half the size of the other one. As I am sexually active, I feel a pain in each of the vas deferens after sex. If I do not have sex for days, they become sensitive. I was treated with Ceftum 500 mg, the first time I had an infection. Now my question is, is there any chance that the infection is still in my body? And, do I need to get examined for it? For the time being, please suggest some precautionary antibiotics for my relief.', 'output': 'Hello, Welcome to Chat Doctor forum. I can understand your concern. You had mumps and this is a viral infection known to cause an inflammation of the testis in some cases. Take care. For more information consult a sexologist online'}
Column names: ['instruction', 'input',

Define the Matching Function

In [16]:
def find_best_match(user_question, dataset):
    best_score = 0
    best_answer = "Sorry, I couldn't find a relevant answer. Please try rephrasing your question."
    best_question = None

    # Iterate through the dataset with error handling
    try:
        for item in dataset:
            if not isinstance(item, dict):
                continue  # Skip non-dictionary items
            question = item['input']
            score = fuzz.token_sort_ratio(user_question.lower(), question.lower())
            if score > best_score and score > 50:  # Threshold to avoid irrelevant matches
                best_score = score
                best_answer = item['output']
                best_question = question
    except TypeError as e:
        print(f"Error in matching: {e}")
        return best_answer, best_question, best_score

    return best_answer, best_question, best_score

Define the Chatbot Interface

In [21]:
def chatbot():
    print("Welcome to MediBot: ChatDoctor-iCliniq QA! Type 'exit', 'thank you', 'bye', 'quit', 'done', or 'thanks' to quit.")
    exit_words = ['exit', 'thank you', 'bye', 'quit', 'done', 'thanks']
    farewell_messages = [
        "Take care!",
        "See you later!",
        "Stay healthy!",
        "Farewell!",
        "Have a great day!",
        "Until next time!"
    ]
    while True:
        user_input = input("\nYour question: ").strip()
        if user_input.lower() in exit_words:
            print(random.choice(farewell_messages))
            break

        # Find the best matching answer
        answer, matched_question, score = find_best_match(user_input, data)

        # Display the response
        if score > 50:
            print(f"\nMatched Question: {matched_question}")
            print(f"Answer: {answer}")
        else:
            print(f"\n{answer}")

Run the Chatbot

In [24]:
if __name__ == "__main__":
    try:
        chatbot()
    except KeyboardInterrupt:
        print("\nInterrupted by user. Take care!")
        sys.exit(0)

Welcome to MediBot: ChatDoctor-iCliniq QA! Type 'exit', 'thank you', 'bye', 'quit', 'done', or 'thanks' to quit.

Your question: Hello doctor,Some mornings during the week I get a quick flash of numbness over my left temple.

Matched Question: Hello doctor,I have a small red or pink hive from the last fifteen years on my right hand.
Answer: Hello. I have gone through your query and have viewed the image (attachment removed to protect patient identity). Regards. For more information consult a dermatologist online  Take care.

Your question: bye
Have a great day!
